In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
##################################################################

import logging
import torch
import transformers

from rich.console import Console
from rich.theme import Theme

from src.utils import logging_utils

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme)


def rich_print(*args, **kwargs):
    console.print(*args, **kwargs)


logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

2026-03-17 18:40:30 __main__ INFO     torch.__version__='2.10.0+cu128', torch.version.cuda='12.8'
2026-03-17 18:40:30 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=2, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
2026-03-17 18:40:30 __main__ INFO     transformers.__version__='5.3.0'


## Model

In [3]:
from src.models import ModelandTokenizer

mt = ModelandTokenizer(
    model_key="Qwen/Qwen2.5-Coder-32B-Instruct",
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
rich_print(mt)

2026-03-17 18:40:36 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-03-17 18:40:36 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-03-17 18:40:36 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7fe0cd5cb440>
2026-03-17 18:40:36 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7fe0cc5952d0> server_hostname='huggingface.co' timeout=10
2026-03-17 18:40:36 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7fe0cc35a060>
2026-03-17 18:40:36 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-03-17 18:40:36 httpcore.http11 DEBUG    send_request_headers.complete
2026-03-17 18:40:36 http

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-03-17 18:40:58 httpcore.connection DEBUG    close.started
2026-03-17 18:40:58 httpcore.connection DEBUG    close.complete
2026-03-17 18:40:58 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-03-17 18:40:58 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7fe0ca840230>
2026-03-17 18:40:58 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7fe0cc5952d0> server_hostname='huggingface.co' timeout=10
2026-03-17 18:40:58 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7fe0ccd01ca0>
2026-03-17 18:40:58 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-03-17 18:40:58 httpcore.http11 DEBUG    send_request_headers.complete
2026-03-17 18:40:58 httpcore.http11 DEBUG    send_request_body.started request=<Request [b'HE

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 5120)
    (layers): ModuleList(
      (0-63): 64 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=5120, out_features=5120, bias=True)
          (k_proj): Linear(in_features=5120, out_features=1024, bias=True)
          (v_proj): Linear(in_features=5120, out_features=1024, bias=True)
          (o_proj): Linear(in_features=5120, out_features=5120, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=5120, out_features=27648, bias=False)
          (up_proj): Linear(in_features=5120, out_features=27648, bias=False)
          (down_proj): Linear(in_features=27648, out_features=5120, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((5120,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((5120,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((5120,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbedding()
  )
  (lm_head): Linear(in_features=5120, out_features=152064, bias=False)
  (generator): Generator(
    (streamer): Streamer()
  )
)

## Dataset

In [4]:
from src.goals.head_loc_data import HeadLocDataset

ds = HeadLocDataset.load()
rich_print(f"Loaded [bold]{len(ds)}[/bold] predicate samples, [bold]{len(ds.pairs)}[/bold] pairs")
SAMPLE_IDX = 0
sample = ds.samples[SAMPLE_IDX]
rich_print(
    f"  predicate_name : {sample.predicate_name}\n",
    f"  variable_type  : {sample.variable_type}\n",
    f"  variable_name  : {sample.variable_name}\n",
    f"  docstring      : {sample.docstring!r}\n",
    f"  python_incomplete_code:\n{sample.python_incomplete_code}\n",
    f"  cpp_incomplete_code:\n{sample.cpp_incomplete_code}\n",
    f"  unit_tests     : {sample.unit_tests[:3]}..."
)

2026-03-17 18:41:04 src.goals.head_loc_data INFO     Loaded 6 predicates, 6 pairs from /disk/u/arnab/Codes/Projects/goals/data/head_loc


Loaded 6 predicate samples, 6 pairs

predicate_name : has_duplicates
   variable_type  : list
   variable_name  : lst
   docstring      : ''
   python_incomplete_code:
def has_duplicates(lst: list) -> bool:
    return
   cpp_incomplete_code:
bool hasDuplicates(vector<int> lst) { return
   unit_tests     : [([], False), ([1], False), ([1, 2, 3, 4], False)]...

In [5]:
from rich.table import Table

table = Table(title="Predicate Pairs", highlight=False)
table.add_column("Source (C++)", style="cyan")
table.add_column("Destination (Python)", style="green")
table.add_column("Type", style="yellow")
table.add_column("Oracle verified", justify="center")

for pair in ds.pairs:
    table.add_row(
        pair.source_predicate,
        pair.destination_predicate,
        pair.variable_type,
        "[green]YES[/green]" if pair.oracle_verified else "[dim]no[/dim]",
    )

console.print(table)

                          Predicate Pairs                          
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Source (C++)    ┃ Destination (Python) ┃ Type ┃ Oracle verified ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ has_duplicates  │ is_sorted            │ list │       YES       │
│ is_sorted       │ has_duplicates       │ list │       YES       │
│ is_odd          │ is_power_of_two      │ int  │       YES       │
│ is_power_of_two │ is_odd               │ int  │       YES       │
│ is_palindrome   │ is_upper             │ str  │       YES       │
│ is_upper        │ is_palindrome        │ str  │       YES       │
└─────────────────┴──────────────────────┴──────┴─────────────────┘

## Case 1 — Bare Return Pivot Instances

For each pair, compute the top predicted token at `return ` (bare return, no prefix) for
both dest and target.  Keep pairs where the top tokens differ.

Set `include_docstring=True` to insert each sample's `docstring` field between the
def line and `return`, giving the model a semantic hint.

In [6]:
from src.goals.head_loc_pivot import case1_pivot_instances

# include_docstring=False: no docstring hint (default, cleaner signal)
case1_instances = case1_pivot_instances(mt, ds, include_docstring=False)
rich_print(f"[bold]Case 1:[/bold] {len(case1_instances)} diverging pivot instances")
for pi in case1_instances:
    rich_print(
        f"  {pi.destination_predicate} -> {pi.target_predicate}"
        f" | prefix={pi.pivot_prefix!r}"
        f" | dest_top={pi.dest_top_token!r}  targ_top={pi.target_top_token!r}"
        f" | diverges=[green]YES[/green]" if pi.tokens_diverge else " | diverges=[red]no[/red]"
    )

2026-03-17 14:31:33 src.goals.head_loc_pivot DEBUG    Case 1 | is_sorted top=' all' | has_duplicates top=' len' | diverge=True
2026-03-17 14:31:33 src.goals.head_loc_pivot DEBUG    Case 1 | has_duplicates top=' len' | is_sorted top=' all' | diverge=True
2026-03-17 14:31:33 src.goals.head_loc_pivot DEBUG    Case 1 | is_power_of_two top='0' | is_odd top=' n' | diverge=True
2026-03-17 14:31:33 src.goals.head_loc_pivot DEBUG    Case 1 | is_odd top=' n' | is_power_of_two top='0' | diverge=True
2026-03-17 14:31:34 src.goals.head_loc_pivot DEBUG    Case 1 | is_upper top='0' | is_palindrome top=' s' | diverge=True
2026-03-17 14:31:34 src.goals.head_loc_pivot DEBUG    Case 1 | is_palindrome top=' s' | is_upper top='0' | diverge=True
2026-03-17 14:31:34 src.goals.head_loc_pivot INFO     Case 1: 6 diverging pivot instances out of 6 pairs


Case 1: 6 diverging pivot instances

is_sorted -> has_duplicates | prefix='' | dest_top=' all'  targ_top=' len' | diverges=YES

has_duplicates -> is_sorted | prefix='' | dest_top=' len'  targ_top=' all' | diverges=YES

is_power_of_two -> is_odd | prefix='' | dest_top='0'  targ_top=' n' | diverges=YES

is_odd -> is_power_of_two | prefix='' | dest_top=' n'  targ_top='0' | diverges=YES

is_upper -> is_palindrome | prefix='' | dest_top='0'  targ_top=' s' | diverges=YES

is_palindrome -> is_upper | prefix='' | dest_top=' s'  targ_top='0' | diverges=YES

## Case 2 — MCTS-Based Shared Prefix Pivot Instances

MCTS exhaustively explores the generation tree, providing diverse passing
completions for reliable LCP detection. The pipeline is decoupled:

1. **Run MCTS once** (slow, ~200 iters/predicate) — saves results to disk.
2. **Find pivots** (fast, online per model) — reads cached green_completions.

Set  (default) to load cached results from .

In [7]:
from rich.table import Table
from src.goals.head_loc_pivot import run_mcts_for_dataset

mcts_results = run_mcts_for_dataset(
    mt, 
    ds,
    include_docstring = True, 
    overwrite=False, 
    n_iterations=1024, 
    max_depth=32,
    c_puct=1.0,
    min_children_up_to_depth=3,
    min_children=2,
)

table = Table(title="MCTS Results per Predicate", highlight=False)
table.add_column("Predicate", style="cyan")
table.add_column("Green", justify="right", style="green")
table.add_column("Total", justify="right")
table.add_column("Unique completions", justify="right")
for name, res in mcts_results.items():
    counts = res.eval_counts
    table.add_row(
        name,
        str(counts.get("green", 0)),
        str(counts.get("total", 0)),
        str(len(res.green_completions)),
    )
console.print(table)

2026-03-17 14:31:37 src.goals.head_loc_pivot INFO     run_mcts_for_dataset: save_dir=/disk/u/arnab/Codes/Projects/goals/results/localization/mcts/Qwen2.5-Coder-32B-Instruct
2026-03-17 14:31:37 src.goals.head_loc_pivot INFO       has_duplicates: running MCTS (1024 iters)…
2026-03-17 14:31:42 src.goals.sampling DEBUG    MCTS iteration 50/1024, total nodes: 104
2026-03-17 14:31:46 src.goals.sampling DEBUG    MCTS iteration 100/1024, total nodes: 195
2026-03-17 14:31:48 src.goals.sampling DEBUG    MCTS iteration 150/1024, total nodes: 243
2026-03-17 14:31:48 src.goals.sampling DEBUG    MCTS iteration 200/1024, total nodes: 253
2026-03-17 14:31:50 src.goals.sampling DEBUG    MCTS iteration 250/1024, total nodes: 297
2026-03-17 14:31:51 src.goals.sampling DEBUG    MCTS iteration 300/1024, total nodes: 309
2026-03-17 14:31:52 src.goals.sampling DEBUG    MCTS iteration 350/1024, total nodes: 333
2026-03-17 14:31:54 src.goals.sampling DEBUG    MCTS iteration 400/1024, total nodes: 364
2026-03-1

               MCTS Results per Predicate               
┏━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Predicate       ┃ Green ┃ Total ┃ Unique completions ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ has_duplicates  │    12 │   237 │                  6 │
│ is_odd          │    19 │    67 │                  9 │
│ is_palindrome   │     6 │    84 │                  3 │
│ is_power_of_two │     8 │    93 │                  4 │
│ is_sorted       │     9 │    21 │                  5 │
│ is_upper        │     6 │   109 │                  3 │
└─────────────────┴───────┴───────┴────────────────────┘

### Tree Inspection (single predicate)

Run MCTS on one sample, evaluate the tree, walk green leaves,
and display the colored HTML tree inline.

In [ ]:
import os
from src.goals.sampling import MCTSSearch
from src.goals.eval import evaluate_mcts_tree
from src.goals.head_loc_pivot import build_python_prompt, _extract_effective_completions

INSPECT_IDX = 2  # change to inspect a different predicate
inspect_sample = ds.samples[INSPECT_IDX]

prompt = build_python_prompt(mt, inspect_sample, pivot_prefix="")

rich_print(f"'[white]{prompt}[/white]'")

In [ ]:
search = MCTSSearch(
    mt,
    prompt,
    max_depth=32,
    top_k=5,
    temperature=1.0,
    c_puct=1.0,
    min_children_up_to_depth=3,
    min_children=2,
)
search.run(n_iterations=1024, depth_first=True)

In [ ]:
prefix = inspect_sample.python_incomplete_code + " "

eval_counts = evaluate_mcts_tree(
    root=search.root,
    prefix=prefix,
    unit_tests=inspect_sample.unit_tests,
    function_name=inspect_sample.function_name or inspect_sample.predicate_name,
)
rich_print(f"[bold]{inspect_sample.predicate_name}[/bold] eval_counts: {eval_counts}")

In [ ]:
# Walk green leaves and show effective completions
green_completions = _extract_effective_completions(search.root)
rich_print(f"Unique green completions ({len(green_completions)}):")
for c in green_completions:
    rich_print(f"  [green]{c!r}[/green]")

In [ ]:
# Display the colored MCTS tree inline
import tempfile
from IPython.display import HTML, display

with tempfile.NamedTemporaryFile(suffix=".html", delete=False) as _f:
    _tmp = _f.name
search.save_tree(_tmp, show_colors=True)
with open(_tmp) as _f:
    _html = _f.read()
os.unlink(_tmp)
display(HTML(_html))

In [ ]:
# mcts_results["is_palindrome"].__dict__.keys()
mcts_results["is_palindrome"].green_completions

## Contrastive pairs

In [24]:
from src.goals.head_loc_pivot import build_python_prompt

pair = ds.pairs[-2]
dest_sample = ds.get_sample(pair.destination_predicate)
targ_sample = ds.get_sample(pair.source_predicate)

dest_prompt = build_python_prompt(mt, dest_sample, pivot_prefix="")
targ_prompt = build_python_prompt(mt, targ_sample, pivot_prefix="")

rich_print(f'Destination prompt:\n[white]"{dest_prompt}"[/white]')
rich_print(f'Source prompt:\n[white]"{targ_prompt}"[/white]')

Destination prompt:
"<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Complete the Python function below.<|im_end|>
<|im_start|>assistant
<code>
def is_upper(s: str) -> bool:
    return "

Source prompt:
"<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Complete the Python function below.<|im_end|>
<|im_start|>assistant
<code>
def is_palindrome(s: str) -> bool:
    return "

In [25]:
dest_result = mcts_results.get(pair.destination_predicate)
targ_result = mcts_results.get(pair.source_predicate)

dest_passing = dest_result.green_completions
targ_passing = targ_result.green_completions


rich_print(f"Destination passing completions ({len(dest_passing)}): {dest_passing}")
rich_print(f"Target passing completions ({len(targ_passing)}): {targ_passing}")

NameError: name 'mcts_results' is not defined

In [ ]:
from src.goals.head_loc_pivot import _find_stable_lcps

stable_lcps = _find_stable_lcps(mt, dest_passing, targ_passing, min_stable_k=2)
stable_lcps

In [ ]:
from src.goals.head_loc_pivot import case2_pivot_instances

case2_instances = case2_pivot_instances(
    mt, ds, mcts_results, min_stable_k=1, include_docstring=True
)
rich_print(f"[bold]Case 2:[/bold] {len(case2_instances)} diverging pivot instances")
for pi in case2_instances:
    rich_print(
        f"  {pi.destination_predicate} -> {pi.target_predicate}"
        f" | prefix={pi.pivot_prefix!r}"
        f" | dest_top={pi.dest_top_token!r}  targ_top={pi.target_top_token!r}"
    )

In [ ]:
# Combine and deduplicate
seen = set()
deduped = []
for pi in case1_instances + case2_instances:
    key = (pi.destination_predicate, pi.target_predicate, pi.pivot_prefix)
    if key not in seen:
        seen.add(key)
        deduped.append(pi)

rich_print(f"Total: {len(case1_instances) + len(case2_instances)} -> {len(deduped)} after dedup")

# All instances satisfy tokens_diverge by construction
assert all(pi.tokens_diverge for pi in deduped), "Expected all instances to diverge"

# Save
ds.save_pivot_instances(deduped)
rich_print("[green]Saved pivot_instances.json[/green]")

# Round-trip check
loaded = HeadLocDataset.load_pivot_instances()
rich_print(f"Round-trip: loaded {len(loaded)} instances")

## Augmentation

Dest and target are augmented **jointly** so their pivot prefix tokens stay identical.
`augment_pair` applies the same transformation to both samples with a single call.

In [ ]:
from src.goals.head_loc_augment import augment_pair

dest_name = ds.samples[0].predicate_name
target_name = ds.samples[1].predicate_name if len(ds.samples) > 1 else ds.samples[0].predicate_name

dest = ds[dest_name]
target = ds[target_name]

rich_print(f"[bold]Demo pair:[/bold] dest={dest_name!r} ({dest.variable_name!r}), target={target_name!r} ({target.variable_name!r})")

new_var = "word"
d2, t2 = augment_pair(dest, target, "variable_rename", {"new_var": new_var})

rich_print(f"\n[bold]After variable_rename → {new_var!r}:[/bold]")
rich_print(f"  dest before : {dest.python_incomplete_code!r}")
rich_print(f"  dest after  : {d2.python_incomplete_code!r}")
rich_print(f"  targ before : {target.python_incomplete_code!r}")
rich_print(f"  targ after  : {t2.python_incomplete_code!r}")
rich_print(f"  d2.variable_name={d2.variable_name!r}, t2.variable_name={t2.variable_name!r}")
rich_print(f"  augmentation record: {d2.augmentation}")

In [ ]:
from src.goals.head_loc_augment import generate_pair_variants

variants = generate_pair_variants(dest, target)
rich_print(f"[bold]generate_pair_variants:[/bold] {len(variants)} (dest, target) variant pairs")
for i, (dv, tv) in enumerate(variants):
    aug_type = dv.augmentation[-1]["type"] if dv.augmentation else "unknown"
    rich_print(
        f"  [{i}] {aug_type}"
        f" | dest.variable_name={dv.variable_name!r}"
        f" | targ.variable_name={tv.variable_name!r}"
    )

## Case 3 — Oracle-Based Pivot Instances

**Motivation:** MCTS finds LCPs across all passing implementations of the *destination*
predicate.  When the destination has structurally diverse implementations (e.g. `s.isupper()`
and `s == s.upper()`), the LCP collapses to just `s` — missing the deeper divergence point.

**Fix:** Ask an oracle LLM for canonical implementations of each predicate independently,
then cross-compute LCPs between (dest_impl, target_impl) pairs.  This finds longer,
semantically meaningful shared prefixes like `s == s.` where `.upper()` vs `[::-1]`
first diverge.

**Two gates filter oracle pivots:**
1. **Plausibility gate** — LM assigns ≥ `min_plausibility` to the first LCP token at the bare-return position (the model would naturally reach this prefix).
2. **Divergence gate** — `dest_top_token != target_top_token` at `return {lcp}`.


In [ ]:

# ── Inspect oracle pair prompt for the noisy pair ────────────────────────────
from src.goals.head_loc_oracle import (
    build_oracle_pair_prompt,
    get_oracle_lcps,
    get_oracle_implementations,  # kept for standalone inspection
)

INSPECT_PAIR_DEST   = "is_upper"
INSPECT_PAIR_TARGET = "is_palindrome"

dest_sample_ex = ds.get_sample(INSPECT_PAIR_DEST)
targ_sample_ex = ds.get_sample(INSPECT_PAIR_TARGET)

rich_print("[bold]Oracle pair prompt:[/bold]")
rich_print(f"[white]{build_oracle_pair_prompt(dest_sample_ex, targ_sample_ex)}[/white]")


In [ ]:

# ── Oracle-proposed LCPs for the inspected pair ──────────────────────────────
oracle_lcps_ex = get_oracle_lcps(dest_sample_ex, targ_sample_ex, n=5, temperature=0.5)

rich_print(f"[bold]Oracle LCPs for ({INSPECT_PAIR_DEST}, {INSPECT_PAIR_TARGET}):[/bold]")
for lcp in oracle_lcps_ex:
    rich_print(f"  return [yellow]{lcp!r}[/yellow] ...")


In [ ]:

# ── Completion gate smoke-test for each oracle LCP ───────────────────────────
# Manually verify: does the model produce valid code from each oracle prefix?
from src.goals.head_loc_oracle import _completion_gate

nl_ids = set(mt.tokenizer.encode("\n", add_special_tokens=False))
stop_ids = {mt.tokenizer.eos_token_id} | nl_ids

for lcp in oracle_lcps_ex:
    ok = _completion_gate(mt, dest_sample_ex, targ_sample_ex, lcp, max_new_tokens=32)
    marker = "[green]✓ PASS[/green]" if ok else "[red]✗ FAIL[/red]"
    rich_print(f"  {marker}  lcp={lcp!r}")


In [9]:
ds.pairs

[PredicatePair(source_predicate='has_duplicates', destination_predicate='is_sorted', variable_type='list[int]', oracle_verified=True, kl_at_bare_return=6.3303, top_token_diverges_at_bare_return=True),
 PredicatePair(source_predicate='is_sorted', destination_predicate='has_duplicates', variable_type='list[int]', oracle_verified=True, kl_at_bare_return=6.6259, top_token_diverges_at_bare_return=True),
 PredicatePair(source_predicate='is_odd', destination_predicate='is_power_of_two', variable_type='int', oracle_verified=True, kl_at_bare_return=1.4389, top_token_diverges_at_bare_return=True),
 PredicatePair(source_predicate='is_power_of_two', destination_predicate='is_odd', variable_type='int', oracle_verified=True, kl_at_bare_return=1.9636, top_token_diverges_at_bare_return=True),
 PredicatePair(source_predicate='is_palindrome', destination_predicate='is_upper', variable_type='str', oracle_verified=True, kl_at_bare_return=1.5753, top_token_diverges_at_bare_return=True),
 PredicatePair(sour

In [11]:

# ── Full Case 3 run across all pairs ─────────────────────────────────────────
from src.goals.head_loc_oracle import case3_pivot_instances

case3_instances = case3_pivot_instances(
    mt,
    ds,
    include_docstring=True,
    n_oracle=5,
    oracle_temperature=0.5,
    min_stable_k=1,
    max_completion_tokens=32,
    pairs=[ds.pairs[-2]]
)

rich_print(f"[bold]Case 3:[/bold] {len(case3_instances)} diverging pivot instances")
for pi in case3_instances:
    rich_print(
        f"  {pi.destination_predicate} -> {pi.target_predicate}"
        f" | prefix={pi.pivot_prefix!r}"
        f" | dest_top={pi.dest_top_token!r}  targ_top={pi.target_top_token!r}"
    )


2026-03-17 18:43:08 src.goals.head_loc_oracle INFO     Case 3: pair is_upper -> is_palindrome
2026-03-17 18:43:08 src.utils.oracle_llms DEBUG    found cached claude response for 93d897627d877fe7c60d2e791bdabe4a - loading
2026-03-17 18:43:08 src.goals.head_loc_oracle DEBUG    Oracle response for 'is_upper':
- s.isupper()
- s == s.upper()
- all(c.isupper() for c in s if c.isalpha())
- not any(c.islower() for c in s) and any(c.isalpha() for c in s)
- s.upper() == s and bool(s) and any(c.isalpha() for c in s)
2026-03-17 18:43:08 src.goals.head_loc_oracle INFO     Oracle implementations for 'is_upper' (5 unique): ['s.isupper()', 's == s.upper()', 'all(c.isupper() for c in s if c.isalpha())', 'not any(c.islower() for c in s) and any(c.isalpha() for c in s)', 's.upper() == s and bool(s) and any(c.isalpha() for c in s)']
2026-03-17 18:43:08 src.utils.oracle_llms DEBUG    found cached claude response for 95f233572592eb14796a71ae7ff17b23 - loading
2026-03-17 18:43:08 src.goals.head_loc_oracle DE

Case 3: 4 diverging pivot instances

is_upper -> is_palindrome | prefix='s' | dest_top='.is'  targ_top=' =='

is_upper -> is_palindrome | prefix='s == s' | dest_top='.upper'  targ_top='[::-'

is_upper -> is_palindrome | prefix='all' | dest_top='(c'  targ_top='(s'

is_upper -> is_palindrome | prefix='not any' | dest_top='(c'  targ_top='(s'

In [21]:
s = "123"
s == s.upper()

True

In [ ]:

# ── Combine Case 1 + 2 + 3, deduplicate, and save ────────────────────────────
seen = set()
combined = []
for pi in case1_instances + case2_instances + case3_instances:
    key = (pi.destination_predicate, pi.target_predicate, pi.pivot_prefix)
    if key not in seen:
        seen.add(key)
        combined.append(pi)

assert all(pi.tokens_diverge for pi in combined), "All instances must diverge"

rich_print(
    f"Total: Case1={len(case1_instances)} + Case2={len(case2_instances)}"
    f" + Case3={len(case3_instances)} = {len(combined)} after dedup"
)

# Save
ds.save_pivot_instances(combined)
rich_print("[green]Saved pivot_instances.json with Case 3 augmentation[/green]")

loaded = HeadLocDataset.load_pivot_instances()
rich_print(f"Round-trip: loaded {len(loaded)} instances")
